## Predicting review helpfulness (vote column) from review text only - v3
**What was wrong in v2 (found by checking the data):**
1. **The validation set did not look like the test set.** v2 validated on the newest 10% of reviews (base rate 9.8% helpful), but `test_data.csv` spans the same period as the training data (2016-01 to 2018-09) with a **24.7%** base rate. PR-AUC and F1 depend directly on the base rate, so the v2 numbers looked much worse than they were (v2 AUC 0.759 is the same as the earlier random-split notebook, 0.760). The thresholds and the stacker were also fitted on a distribution that the test set does not have.
2. **The cohort label changed nothing.** 77% of the reviews have no vote, so the monthly 75th percentile of votes is 0 (and at most 2 in the few months with many voted reviews). The cohort threshold therefore never rose above 2, which is exactly the raw threshold (no review has exactly 1 vote), and the `raw` and `chosen` columns of the v2 table were identical.
3. **`test_data.csv` does have `vote`, `unixReviewTime` and `asin`** (4,933 of 20,000 rows have a vote). That means the final model can be scored on a real, untouched test set instead of only on validation data.

**Changes in v3**
* Random train / validation split (matches the test set), label = `vote >= 2` (no review has exactly 1 vote in this data).
* Models still see **review text only** (the product id and the time are never model inputs). They are used only for diagnostics.
* Validation is used for model selection, threshold and stacker fitting. **The labelled test set is used once, for the final report.**
* Reports also include **lift** (precision in the top 10% divided by the base rate), which is comparable across different base rates.

In [1]:
!pip -q install gdown
import os, re, html, time, random, json
import numpy as np, pandas as pd, torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

CFG = dict(
    model="distilroberta-base",
    n_train=300_000, n_val=40_000,
    epochs=1, bs=32, lr=3e-5, warmup=0.06, weight_decay=0.01,
    max_len=256, max_summary=32, head_frac=0.7,
    eval_every=2000, eval_n=10_000, seed=42,
    use_summary=False,            # task says text only; set True if summary is allowed
    min_votes=2,                  # helpful = vote >= 2 (no review has exactly 1 vote in this data)
    pos_weight_power=0.5,         # positive class weight = (neg/pos)**power
    tfidf_max_features=300_000,
)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

device: cuda Tesla T4


In [2]:
import gdown
from google.colab import drive

# checkpoints, best weights and predictions are kept on Drive so a disconnect does not lose the training
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/review_helpfulness"
os.makedirs(OUT, exist_ok=True)

os.makedirs("data", exist_ok=True)
FILES = {"train_data.csv": "1-AlW7oNJHaqi3xk_9dWHUS52Dzl_FmFW",
         "test_data.csv":  "1-8TsrqTRFP-q9TM-6HinhO0ZVXFHq9TB"}
for name, fid in FILES.items():
    if not os.path.exists(f"data/{name}"):
        gdown.download(id=fid, output=f"data/{name}", quiet=False)

Mounted at /content/drive


Downloading...
From (original): https://drive.google.com/uc?id=1-AlW7oNJHaqi3xk_9dWHUS52Dzl_FmFW
From (redirected): https://drive.google.com/uc?id=1-AlW7oNJHaqi3xk_9dWHUS52Dzl_FmFW&confirm=t&uuid=8a7f631e-ea3e-4193-8c89-b2399b01bdba
To: /content/data/train_data.csv
100%|██████████| 635M/635M [00:08<00:00, 72.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-8TsrqTRFP-q9TM-6HinhO0ZVXFHq9TB
To: /content/data/test_data.csv
100%|██████████| 15.6M/15.6M [00:00<00:00, 36.2MB/s]


## 1. Labels and the train / test relationship

In [3]:
hdr = {c.lower(): c for c in pd.read_csv("data/train_data.csv", nrows=0).columns}
need = ["reviewtext", "summary", "vote", "unixreviewtime", "asin"]
missing = [c for c in need if c not in hdr]
assert not missing, f"missing columns in train_data.csv: {missing}; available: {list(hdr.values())}"
df = pd.read_csv("data/train_data.csv", usecols=[hdr[c] for c in need], low_memory=False)
df = df.rename(columns={hdr[c]: n for c, n in zip(need, ["reviewText", "summary", "vote", "unixReviewTime", "asin"])})
test_df = pd.read_csv("data/test_data.csv", low_memory=False)       # the test file also has vote / unixReviewTime / asin
df = df.dropna(subset=["reviewText", "unixReviewTime"]).reset_index(drop=True)

def parse_votes(s):
    # vote is often a string such as "1,234"; NaN means the review received no votes
    return pd.to_numeric(s.astype(str).str.replace(",", "", regex=False), errors="coerce").fillna(0)

df["vote"], test_df["vote"] = parse_votes(df["vote"]), parse_votes(test_df["vote"])
print("vote value counts (train):"); print(df["vote"].value_counts().sort_index().head(6).to_string())
df["y"] = (df["vote"] >= CFG["min_votes"]).astype(int)
test_df["y"] = (test_df["vote"] >= CFG["min_votes"]).astype(int)
for d in (df, test_df):
    d["month"] = pd.to_datetime(d["unixReviewTime"], unit="s").dt.strftime("%Y-%m")
print(f"positive rate - train pool: {df.y.mean():.3f} | test: {test_df.y.mean():.3f}")

by_month = pd.DataFrame({"train_rate": df.groupby("month").y.mean(), "test_rate": test_df.groupby("month").y.mean(),
                         "train_share": df.month.value_counts(normalize=True).sort_index(),
                         "test_share": test_df.month.value_counts(normalize=True).sort_index()})
fig = make_subplots(rows=1, cols=2, subplot_titles=["Share of helpful reviews by month", "Share of reviews by month"])
for col, (a, b) in enumerate([("train_rate", "test_rate"), ("train_share", "test_share")], start=1):
    for name, color in [(a, "#1f77b4"), (b, "#ff7f0e")]:
        fig.add_trace(go.Scatter(x=by_month.index, y=by_month[name], mode="lines+markers", name=name,
                                 line=dict(color=color)), row=1, col=col)
fig.update_xaxes(tickangle=60)
fig.update_layout(height=380, width=1100)
fig.show()

vote value counts (train):
vote
0.0    647476
2.0     68047
3.0     35503
4.0     20755
5.0     13617
6.0      9523
positive rate - train pool: 0.228 | test: 0.247


The test set covers the same months as the training data, so a **random** split (not a temporal one) gives a validation set that matches it. The helpful rate falls for the newest months because those reviews have had little time to collect votes; this is a property of the data that train and test share.

## 2. Text preprocessing, features and split

In [4]:
_TAG, _URL = re.compile(r"<[^>]+>"), re.compile(r"https?://\S+|www\.\S+")
_REPEAT, _SPACES = re.compile(r"([A-Za-z])\1{3,}"), re.compile(r"\s+")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = _TAG.sub(" ", text)
    text = _URL.sub(" url ", text)
    text = _REPEAT.sub(r"\1\1\1", text)
    return _SPACES.sub(" ", text).strip()

def text_features(series):
    """Simple length / style features computed from the raw review text only"""
    s = series.fillna("").astype(str)
    f = pd.DataFrame(index=s.index)
    f["n_chars"] = s.str.len()
    f["n_words"] = s.str.split().str.len().fillna(0)
    f["n_sent"] = s.str.count(r"[.!?]+")
    f["avg_word_len"] = f["n_chars"] / f["n_words"].clip(lower=1)
    f["n_digits"] = s.str.count(r"\d")
    f["upper_ratio"] = s.str.count(r"[A-Z]") / f["n_chars"].clip(lower=1)
    f["n_excl"] = s.str.count("!")
    f["n_quest"] = s.str.count(r"\?")
    f["has_url"] = s.str.contains(r"https?://|www\.", regex=True).astype(int)
    return f.reset_index(drop=True)

tok = AutoTokenizer.from_pretrained(CFG["model"])

def encode_review(summary, review):
    r_ids = tok.encode(clean_text(review), add_special_tokens=False)
    if CFG["use_summary"]:
        s_ids = tok.encode(clean_text(summary), add_special_tokens=False)[: CFG["max_summary"]]
        head_tokens, n_special, n_summary = [tok.cls_token_id] + s_ids + [tok.sep_token_id] * 2, 4, len(s_ids)
    else:
        head_tokens, n_special, n_summary = [tok.cls_token_id], 2, 0
    budget = CFG["max_len"] - n_special - n_summary
    if len(r_ids) > budget:                                   # keep head + tail
        head = int(budget * CFG["head_frac"]); tail = budget - head
        r_ids = r_ids[:head] + (r_ids[-tail:] if tail > 0 else [])
    return head_tokens + r_ids + [tok.sep_token_id]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [5]:
# Random split: the test set spans the same period as the training data and has a similar base rate
rng = np.random.RandomState(CFG["seed"]); perm = rng.permutation(len(df))
val_df = df.iloc[perm[: CFG["n_val"]]].reset_index(drop=True)
tr_df = df.iloc[perm[CFG["n_val"]: CFG["n_val"] + CFG["n_train"]]].reset_index(drop=True)
tr_y, val_y, te_y = tr_df["y"].values, val_df["y"].values, test_df["y"].values
print(f"positive rate - train: {tr_y.mean():.3f} | val: {val_y.mean():.3f} | test: {te_y.mean():.3f}")

t0 = time.time()
enc = lambda d: [encode_review(s, r) for s, r in zip(d["summary"], d["reviewText"])]
tr_ids, val_ids, te_ids = enc(tr_df), enc(val_df), enc(test_df)
tr_feats, val_feats, te_feats = (text_features(d["reviewText"]) for d in (tr_df, val_df, test_df))
tr_txt, val_txt, te_txt = (d["reviewText"].map(clean_text) for d in (tr_df, val_df, test_df))
print(f"train {len(tr_ids)} | val {len(val_ids)} | test {len(te_ids)} | prepared in {time.time() - t0:.0f}s")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (747 > 512). Running this sequence through the model will result in indexing errors


positive rate - train: 0.229 | val: 0.227 | test: 0.247
train 300000 | val 40000 | test 20000 | prepared in 279s


## 3. Baselines
* **Length-only:** gradient boosting on the simple length/style features.
* **TF-IDF + logistic regression:** word and bigram features.

These show how much of the transformer's score comes from text length or simple word usage.

In [6]:
def scores(y, p):
    return roc_auc_score(y, p), average_precision_score(y, p)

t0 = time.time()
len_model = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.1, max_iter=200, random_state=CFG["seed"]).fit(tr_feats, tr_y)
p_len = len_model.predict_proba(val_feats)[:, 1]
print("length-only    AUC %.4f | PR-AUC %.4f" % scores(val_y, p_len))

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=CFG["tfidf_max_features"], sublinear_tf=True, dtype=np.float32)
Xtr = tfidf.fit_transform(tr_txt); Xval = tfidf.transform(val_txt)
lr = LogisticRegression(C=1.0, max_iter=200).fit(Xtr, tr_y)
p_tfidf = lr.predict_proba(Xval)[:, 1]
print("tfidf + LR     AUC %.4f | PR-AUC %.4f" % scores(val_y, p_tfidf))
print(f"base rate (random guess) PR-AUC: {val_y.mean():.4f} | baselines took {time.time() - t0:.0f}s")

length-only    AUC 0.6986 | PR-AUC 0.4269
tfidf + LR     AUC 0.7135 | PR-AUC 0.4517
base rate (random guess) PR-AUC: 0.2272 | baselines took 130s


## 4. Transformer
DistilRoBERTa with 2 classes and a mildly weighted cross-entropy. Training uses one epoch because v1 stopped improving early. The best checkpoint is chosen by AUC on a subset of the validation data (note: this is the same data later used for stacking, so the stacked score can be very slightly optimistic). At every evaluation a checkpoint is saved to Google Drive, so if Colab disconnects, re-running the notebook continues training from the last checkpoint instead of starting over.

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(CFG["model"], num_labels=2).to(dev)

def collate(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, : len(s)] = torch.tensor(s); mask[i, : len(s)] = 1
    return ids.to(dev), mask.to(dev)

def train_batches(bs, ep):
    # the order depends only on the seed and the epoch, so a resumed run gets the same batches
    rng = np.random.RandomState(CFG["seed"] + ep)
    idx = rng.permutation(len(tr_ids)); chunk = bs * 50; out = []
    for c in range(0, len(idx), chunk):
        part = sorted(idx[c: c + chunk], key=lambda i: len(tr_ids[i]))
        out += [part[b: b + bs] for b in range(0, len(part), bs)]
    rng.shuffle(out); return out

@torch.no_grad()
def predict_prob(ids_list, bs=128):
    """Probability of the helpful class for each sample"""
    model.eval(); order = np.argsort([len(s) for s in ids_list]); out = np.zeros(len(ids_list), np.float32)
    for b in range(0, len(order), bs):
        sel = order[b: b + bs]; x, m = collate([ids_list[i] for i in sel])
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(input_ids=x, attention_mask=m).logits.float()
        out[sel] = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
    model.train(); return out

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
pos = tr_y.mean(); pos_w = ((1 - pos) / pos) ** CFG["pos_weight_power"]
print(f"positive class weight: {pos_w:.2f}")
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, pos_w], device=dev, dtype=torch.float32))

no_decay = ["bias", "LayerNorm.weight"]
groups = [
    {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": CFG["weight_decay"]},
    {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0},
]
opt = torch.optim.AdamW(groups, lr=CFG["lr"])
total = ((len(tr_ids) + CFG["bs"] - 1) // CFG["bs"]) * CFG["epochs"]
sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * total), total)
scaler = torch.amp.GradScaler()

CKPT, BEST, HIST = f"{OUT}/checkpoint.pt", f"{OUT}/best_helpful.pt", f"{OUT}/train_history.json"

def save_safe(obj, path):
    # write to a temporary file first, so a disconnect in the middle of saving does not break the old file
    torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path)

n_ev = min(CFG["eval_n"], len(val_ids))
best, step, hist = -1.0, 0, []
if os.path.exists(HIST):
    with open(HIST) as f:
        saved = json.load(f)
    best, hist = saved["best"], [tuple(h) for h in saved["hist"]]
    print("training already finished in an earlier run; using the saved best weights")
elif os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=dev)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    step, best, hist = ck["step"], ck["best"], ck["hist"]
    del ck; torch.cuda.empty_cache()
    print(f"resumed at step {step}/{total}")

start_step, t0 = step, time.time()
model.train()
for ep in range(CFG["epochs"]):
    if os.path.exists(HIST):
        break
    run = 0.0
    batches = train_batches(CFG["bs"], ep)
    for k, bidx in enumerate(batches):
        if ep * len(batches) + k < step:          # already trained before the disconnect
            continue
        x, m = collate([tr_ids[i] for i in bidx]); y = torch.tensor(tr_y[bidx], device=dev)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(input_ids=x, attention_mask=m).logits
        loss = loss_fn(logits.float(), y)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        step += 1; run += loss.item()
        if step % 200 == 0:
            el = time.time() - t0
            eta = (total - step) * el / (step - start_step)
            print(f"ep {ep} step {step}/{total} loss {run / 200:.4f} | {el / 60:.1f}m, eta {eta / 60:.1f}m"); run = 0.0
        if step % CFG["eval_every"] == 0 or step == total:
            pv = predict_prob(val_ids[:n_ev])
            auc, ap = map(float, scores(val_y[:n_ev], pv))
            hist.append((step, auc)); print(f"=== step {step}: val AUC = {auc:.4f} | PR-AUC = {ap:.4f}")
            if auc > best:
                best = auc; save_safe(model.state_dict(), BEST)
            save_safe({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                       "scaler": scaler.state_dict(), "step": step, "best": best, "hist": hist}, CKPT)

if not os.path.exists(HIST):
    with open(HIST, "w") as f:
        json.dump({"best": best, "hist": hist}, f)
    os.remove(CKPT)                               # about 1 GB; only the best weights are needed from here on
print("best val AUC (subset):", best)
fig = go.Figure(go.Scatter(x=[h[0] for h in hist], y=[h[1] for h in hist], mode="lines+markers"))
fig.update_layout(title="Validation curve", xaxis_title="step", yaxis_title="val AUC", height=380, width=700)
fig.show()

positive class weight: 1.84
ep 0 step 200/9375 loss 0.6361 | 0.4m, eta 18.6m
ep 0 step 400/9375 loss 0.5932 | 0.8m, eta 17.3m
ep 0 step 600/9375 loss 0.5847 | 1.1m, eta 16.5m
ep 0 step 800/9375 loss 0.5890 | 1.5m, eta 16.3m
ep 0 step 1000/9375 loss 0.5737 | 1.9m, eta 16.0m
ep 0 step 1200/9375 loss 0.5842 | 2.3m, eta 15.6m
ep 0 step 1400/9375 loss 0.5897 | 2.7m, eta 15.4m
ep 0 step 1600/9375 loss 0.5858 | 3.1m, eta 15.1m
ep 0 step 1800/9375 loss 0.5706 | 3.5m, eta 14.6m
ep 0 step 2000/9375 loss 0.5932 | 3.9m, eta 14.2m
=== step 2000: val AUC = 0.7171 | PR-AUC = 0.4278
ep 0 step 2200/9375 loss 0.5722 | 4.5m, eta 14.7m
ep 0 step 2400/9375 loss 0.5757 | 4.9m, eta 14.2m
ep 0 step 2600/9375 loss 0.5932 | 5.3m, eta 13.7m
ep 0 step 2800/9375 loss 0.5819 | 5.7m, eta 13.3m
ep 0 step 3000/9375 loss 0.5869 | 6.1m, eta 12.9m
ep 0 step 3200/9375 loss 0.5821 | 6.5m, eta 12.5m
ep 0 step 3400/9375 loss 0.5660 | 6.8m, eta 12.0m
ep 0 step 3600/9375 loss 0.5802 | 7.2m, eta 11.6m
ep 0 step 3800/9375 loss 0

## 5. Stacking and model comparison (validation set)
The stacker sees: transformer probability, TF-IDF probability and the length/style features. On the validation set it is evaluated with 5-fold out-of-fold predictions, so no row is scored by a stacker that was fitted on it. All four models are compared on the same validation rows. The test set is not touched until section 6.

In [9]:
model.load_state_dict(torch.load(BEST, map_location=dev))
p_trf = predict_prob(val_ids)

def stack_matrix(p_trf, p_tfidf, feats):
    X = feats.copy().reset_index(drop=True)
    X["p_transformer"] = p_trf; X["p_tfidf"] = p_tfidf
    return X

Xs_val = stack_matrix(p_trf, p_tfidf, val_feats)
stacker = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=250, l2_regularization=1.0, random_state=CFG["seed"])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG["seed"])
p_stack = cross_val_predict(stacker, Xs_val, val_y, cv=skf, method="predict_proba")[:, 1]

rows = [(n, *scores(val_y, p)) for n, p in [("length-only", p_len), ("tfidf + LR", p_tfidf), ("transformer", p_trf), ("stacked (OOF)", p_stack)]]
res = pd.DataFrame(rows, columns=["model", "AUC", "PR-AUC"]).round(4)
print(f"validation rows: {len(val_y)} | base rate: {val_y.mean():.4f}")
print(res.to_string(index=False))

validation rows: 40000 | base rate: 0.2272
        model    AUC  PR-AUC
  length-only 0.6986  0.4269
   tfidf + LR 0.7135  0.4517
  transformer 0.7273  0.4523
stacked (OOF) 0.7306  0.4667


In [10]:
# Decision thresholds are tuned on validation only (out-of-fold scores for the stacker)
def best_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-9, None)
    return float(thr[np.argmax(f1)])

THR_TRF, THR_STACK = best_threshold(val_y, p_trf), best_threshold(val_y, p_stack)
print(f"thresholds tuned on validation: transformer {THR_TRF:.3f} | stacked {THR_STACK:.3f}")

thresholds tuned on validation: transformer 0.379 | stacked 0.239


In [11]:
# FINAL EVALUATION on the labelled test set (used once; nothing above was tuned on it)
stacker.fit(Xs_val, val_y)
te_p_len = len_model.predict_proba(te_feats)[:, 1]
te_p_tfidf = lr.predict_proba(tfidf.transform(te_txt))[:, 1]
te_p_trf = predict_prob(te_ids)
te_p_stack = stacker.predict_proba(stack_matrix(te_p_trf, te_p_tfidf, te_feats))[:, 1]

def lift_top10(y, p):
    top = np.argsort(-p)[: int(0.1 * len(p))]
    return y[top].mean() / y.mean()

test_rows = [(n, *scores(te_y, p), lift_top10(te_y, p)) for n, p in
             [("length-only", te_p_len), ("tfidf + LR", te_p_tfidf), ("transformer", te_p_trf), ("stacked", te_p_stack)]]
test_res = pd.DataFrame(test_rows, columns=["model", "AUC", "PR-AUC", "lift@top10%"]).round(4)
print(f"TEST rows: {len(te_y)} | base rate: {te_y.mean():.4f} (random-guess PR-AUC)")
print(test_res.to_string(index=False))

for name, p, t in [("transformer", te_p_trf, THR_TRF), ("stacked", te_p_stack, THR_STACK)]:
    print(f"\n##### {name} on TEST | threshold from validation: {t:.3f}")
    print(classification_report(te_y, (p >= t).astype(int), target_names=["not helpful", "helpful"], digits=3))

TEST rows: 20000 | base rate: 0.2467 (random-guess PR-AUC)
      model    AUC  PR-AUC  lift@top10%
length-only 0.6986  0.4488       2.2137
 tfidf + LR 0.7150  0.4728       2.3738
transformer 0.7291  0.4758       2.3839
    stacked 0.7326  0.4917       2.4083

##### transformer on TEST | threshold from validation: 0.379
              precision    recall  f1-score   support

 not helpful      0.855     0.712     0.777     15067
     helpful      0.418     0.630     0.502      4933

    accuracy                          0.692     20000
   macro avg      0.636     0.671     0.640     20000
weighted avg      0.747     0.692     0.709     20000


##### stacked on TEST | threshold from validation: 0.239
              precision    recall  f1-score   support

 not helpful      0.862     0.686     0.764     15067
     helpful      0.409     0.666     0.507      4933

    accuracy                          0.681     20000
   macro avg      0.636     0.676     0.635     20000
weighted avg      0.75

## 6. Where does the model work? (test set) and saved predictions

In [12]:
# Where does the model work? By publication month and by seen / unseen product (test set)
v = test_df.assign(p_trf=te_p_trf, p_stack=te_p_stack)
rows = []
for mth, g in v.groupby("month"):
    if g.y.nunique() == 2 and len(g) >= 300:
        rows.append({"month": mth, "n": len(g), "pos_rate": round(g.y.mean(), 3),
                     "AUC transformer": round(roc_auc_score(g.y, g.p_trf), 4), "AUC stacked": round(roc_auc_score(g.y, g.p_stack), 4)})
print(pd.DataFrame(rows).to_string(index=False))

seen = v["asin"].isin(set(tr_df["asin"]))
for name, g in [("product seen in training sample", v[seen]), ("product NOT seen in training sample", v[~seen])]:
    if len(g) >= 300 and g.y.nunique() == 2:
        print(f"\n{name}: n={len(g)}, pos_rate={g.y.mean():.3f}, AUC transformer={roc_auc_score(g.y, g.p_trf):.4f}, AUC stacked={roc_auc_score(g.y, g.p_stack):.4f}")

out = pd.DataFrame({"helpful_prob": te_p_stack.round(5), "helpful": (te_p_stack >= THR_STACK).astype(int), "actual_helpful": te_y})
out.to_csv(f"{OUT}/helpful_predictions.csv", index=False)
print("\npredicted helpful share:", round(out.helpful.mean(), 3), "| test base rate:", round(te_y.mean(), 3))
try:
    from google.colab import files; files.download(f"{OUT}/helpful_predictions.csv")
except ImportError:
    pass

  month    n  pos_rate  AUC transformer  AUC stacked
2016-01 1113     0.278           0.7622       0.7587
2016-02  947     0.279           0.6893       0.6953
2016-03  949     0.281           0.7217       0.7349
2016-04  919     0.329           0.7372       0.7448
2016-05  969     0.284           0.7661       0.7638
2016-06  885     0.284           0.7430       0.7453
2016-07  923     0.282           0.7465       0.7448
2016-08  983     0.264           0.7598       0.7587
2016-09  870     0.277           0.6657       0.6763
2016-10  808     0.298           0.7466       0.7493
2016-11  692     0.275           0.7222       0.7239
2016-12  841     0.234           0.7303       0.7378
2017-01  833     0.250           0.7490       0.7507
2017-02  660     0.247           0.7070       0.7025
2017-03  693     0.287           0.6996       0.7013
2017-04  592     0.294           0.7146       0.7238
2017-05  545     0.251           0.7196       0.7235
2017-06  495     0.299           0.6792       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Diagnostic: how much of the label is explained without reading the text?
These scores are **not** used by any model above (the models see review text only). They explain why the text models stop around AUC 0.75: a large part of "helpful" is determined by length and by how much exposure the product gets.

In [13]:
m = tr_df.y.mean(); g = tr_df.groupby("asin").y.agg(["sum", "count"]); k = 5
prior = (g["sum"] + k * m) / (g["count"] + k)                         # smoothed helpful rate of the product in the training sample
sc = test_df["asin"].map(prior).fillna(m); seen_t = test_df["asin"].isin(prior.index)
print(f"product-only score (no text) on TEST: AUC {roc_auc_score(te_y, sc):.4f} | on seen products only {roc_auc_score(te_y[seen_t], sc[seen_t]):.4f} | seen share {seen_t.mean():.3f}")
print(f"review length only (n_chars) on TEST: AUC {roc_auc_score(te_y, te_feats['n_chars']):.4f}")
print(f"transformer on TEST: AUC {roc_auc_score(te_y, te_p_trf):.4f} | stacked: AUC {roc_auc_score(te_y, te_p_stack):.4f}")

product-only score (no text) on TEST: AUC 0.6528 | on seen products only 0.6622 | seen share 0.904
review length only (n_chars) on TEST: AUC 0.6918
transformer on TEST: AUC 0.7291 | stacked: AUC 0.7326
